In [2]:
import os
import pandas as pd
from pathlib import Path
from PIL import Image
import numpy as np

# Go up one level from notebooks/ to Brain_MRI/, then into data/raw/
data_path = Path("../data/raw")

print(f"Looking for data at: {data_path.absolute()}")
print(f"Exists: {data_path.exists()}")

class_counts = {}
for cls_dir in data_path.iterdir():
    if cls_dir.is_dir():
        count = len(list(cls_dir.glob("*.jpg")))
        class_counts[cls_dir.name] = count

print(pd.DataFrame(list(class_counts.items()), columns=["Class", "Count"]))

# Check image dimensions
img_path = list(data_path.glob("*/*.jpg"))[0]
img = Image.open(img_path)
print(f"Image size: {img.size}")

Looking for data at: e:\programming\Brain_MRI\notebooks\..\data\raw
Exists: True
                      Class  Count
0            Astrocytoma T1    396
1          Astrocytoma T1C+    441
2            Astrocytoma T2    281
3             Ependymoma T1    313
4           Ependymoma T1C+    374
5             Ependymoma T2    350
6                 Glioma T1    522
7               Glioma T1C+    549
8                 Glioma T2    405
9     Hemangiopericytoma T1    178
10  Hemangiopericytoma T1C+    307
11    Hemangiopericytoma T2    118
12            Meningioma T1    636
13          Meningioma T1C+    977
14            Meningioma T2    458
15           Neurocytoma T1    190
16         Neurocytoma T1C+    254
17           Neurocytoma T2    174
18                Normal T1    415
19              Normal T1C+    264
20                Normal T2    379
21     Oligodendroglioma T1    221
22   Oligodendroglioma T1C+    192
23     Oligodendroglioma T2    137
24                 Other T1    412
25       

C:\Users\anish\AppData\Local\Temp\ipykernel_10428\4135993530.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [6]:
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path("../src").resolve()))

# Import
from brain_data import prepare_data_loaders

# Load data
train_loader, val_loader, test_loader, class_weights, info = prepare_data_loaders(
    data_dir="../data/raw",
    batch_size=32,
    img_size=224,
)

print("\nDataset Info:")
for key, val in info.items():
    if key != 'class_names':
        print(f"{key}: {val}")

# Test a batch
images, labels = next(iter(train_loader))
print(f"\nBatch shape: {images.shape}")
print(f"Label shape: {labels.shape}")
print(f"Class weights shape: {class_weights.shape}")

Found 30 classes
Astrocytoma T1                 |  792 images
Astrocytoma T1C+               |  882 images
Astrocytoma T2                 |  562 images
Ependymoma T1                  |  626 images
Ependymoma T1C+                |  748 images
Ependymoma T2                  |  700 images
Glioma T1                      | 1044 images
Glioma T1C+                    | 1098 images
Glioma T2                      |  810 images
Hemangiopericytoma T1          |  356 images
Hemangiopericytoma T1C+        |  614 images
Hemangiopericytoma T2          |  236 images
Meningioma T1                  | 1272 images
Meningioma T1C+                | 1954 images
Meningioma T2                  |  916 images
Neurocytoma T1                 |  380 images
Neurocytoma T1C+               |  508 images
Neurocytoma T2                 |  348 images
Normal T1                      |  830 images
Normal T1C+                    |  528 images
Normal T2                      |  758 images
Oligodendroglioma T1           |  442 

In [1]:
import sys
from pathlib import Path
import torch

sys.path.insert(0, str(Path("../src").resolve()))

from brain_data import prepare_data_loaders
from models import BaselineCNN
from train import train_model

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load data
print("\n" + "="*60)
print("Loading data...")
print("="*60)
train_loader, val_loader, test_loader, class_weights, info = prepare_data_loaders(
    data_dir="../data/raw",
    batch_size=32,
    img_size=224,
)

# Create model
print("\nCreating model...")
model = BaselineCNN(num_classes=info['num_classes'], dropout=0.5)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Train
print("\n" + "="*60)
print("Starting training...")
print("="*60)

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=30,
    lr=1e-3,
    device=device,
    class_weights=class_weights,
    save_dir="../models",
    model_name="baseline_cnn"
)

print("\nTraining complete!")
print(f"Best validation accuracy: {max(history['val_acc']):.2f}%")

Using device: cuda

Loading data...
Found 30 classes
Astrocytoma T1                 |  792 images
Astrocytoma T1C+               |  882 images
Astrocytoma T2                 |  562 images
Ependymoma T1                  |  626 images
Ependymoma T1C+                |  748 images
Ependymoma T2                  |  700 images
Glioma T1                      | 1044 images
Glioma T1C+                    | 1098 images
Glioma T2                      |  810 images
Hemangiopericytoma T1          |  356 images
Hemangiopericytoma T1C+        |  614 images
Hemangiopericytoma T2          |  236 images
Meningioma T1                  | 1272 images
Meningioma T1C+                | 1954 images
Meningioma T2                  |  916 images
Neurocytoma T1                 |  380 images
Neurocytoma T1C+               |  508 images
Neurocytoma T2                 |  348 images
Normal T1                      |  830 images
Normal T1C+                    |  528 images
Normal T2                      |  758 images
Ol

c:\Users\anish\anaconda3\envs\gridlock\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(



Epoch [1/30]


Train Loss: 3.3546 | Train Acc: 4.67%
Val Loss: 3.2600 | Val Acc: 8.23%
✓ Best model saved: ..\models\baseline_cnn_best.pth

Epoch [2/30]


Train Loss: 3.2358 | Train Acc: 6.38%
Val Loss: 3.1172 | Val Acc: 6.28%

Epoch [3/30]


...

Train Loss: 2.2256 | Train Acc: 22.73%
Val Loss: 1.9672 | Val Acc: 30.06%

Epoch [27/30]


Train Loss: 2.1827 | Train Acc: 23.56%
Val Loss: 1.8881 | Val Acc: 32.60%
✓ Best model saved: ..\models\baseline_cnn_best.pth

Epoch [28/30]


Train Loss: 2.1488 | Train Acc: 24.25%
Val Loss: 1.8879 | Val Acc: 31.15%

Epoch [29/30]


Train Loss: 2.1516 | Train Acc: 24.46%
Val Loss: 1.8700 | Val Acc: 31.53%

Epoch [30/30]


Train Loss: 2.1376 | Train Acc: 23.86%
Val Loss: 1.8505 | Val Acc: 34.34%
✓ Best model saved: ..\models\baseline_cnn_best.pth

Best validation accuracy: 34.34% (Epoch 30)

Training complete!
Best validation accuracy: 34.34%


In [2]:
import sys
from pathlib import Path
import torch

sys.path.insert(0, str(Path("../src").resolve()))

from brain_data import prepare_data_loaders
from models_tl import EfficientNetTransfer
from train import train_model

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load data (if not already loaded, load it again)
print("\n" + "="*60)
print("Loading data...")
print("="*60)
train_loader, val_loader, test_loader, class_weights, info = prepare_data_loaders(
    data_dir="../data/raw",
    batch_size=32,
    img_size=224,
)

# Create transfer learning model
print("\nCreating EfficientNet-B0 model (pretrained)...")
model = EfficientNetTransfer(num_classes=info['num_classes'], freeze_backbone=True, dropout=0.5)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Train
print("\n" + "="*60)
print("Starting transfer learning training...")
print("="*60)

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=20,  # Fewer epochs for transfer learning
    lr=1e-4,    # Lower learning rate for fine-tuning
    device=device,
    class_weights=class_weights,
    save_dir="../models",
    model_name="efficientnet_frozen"
)

print("\nTransfer learning training complete!")
print(f"Best validation accuracy: {max(history['val_acc']):.2f}%")

Using device: cuda

Loading data...
Found 30 classes
Astrocytoma T1                 |  792 images
Astrocytoma T1C+               |  882 images
Astrocytoma T2                 |  562 images
Ependymoma T1                  |  626 images
Ependymoma T1C+                |  748 images
Ependymoma T2                  |  700 images
Glioma T1                      | 1044 images
Glioma T1C+                    | 1098 images
Glioma T2                      |  810 images
Hemangiopericytoma T1          |  356 images
Hemangiopericytoma T1C+        |  614 images
Hemangiopericytoma T2          |  236 images
Meningioma T1                  | 1272 images
Meningioma T1C+                | 1954 images
Meningioma T2                  |  916 images
Neurocytoma T1                 |  380 images
Neurocytoma T1C+               |  508 images
Neurocytoma T2                 |  348 images
Normal T1                      |  830 images
Normal T1C+                    |  528 images
Normal T2                      |  758 images
Ol

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\anish/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:01<00:00, 15.6MB/s]
c:\Users\anish\anaconda3\envs\gridlock\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Total parameters: 4,343,194
Trainable parameters: 335,646

Starting transfer learning training...

Epoch [1/20]


Train Loss: 3.1916 | Train Acc: 12.52%
Val Loss: 2.8888 | Val Acc: 27.91%
✓ Best model saved: ..\models\efficientnet_frozen_best.pth

Epoch [2/20]


Train Loss: 2.6581 | Train Acc: 23.58%
Val Loss: 2.4182 | Val Acc: 33.27%
✓ Best model saved: ..\models\efficientnet_frozen_best.pth

Epoch [3/20]


...

Train Loss: 1.6866 | Train Acc: 43.88%
Val Loss: 1.3773 | Val Acc: 55.37%
✓ Best model saved: ..\models\efficientnet_frozen_best.pth

Epoch [17/20]


Train Loss: 1.6722 | Train Acc: 44.29%
Val Loss: 1.3571 | Val Acc: 56.64%
✓ Best model saved: ..\models\efficientnet_frozen_best.pth

Epoch [18/20]


Train Loss: 1.6508 | Train Acc: 44.92%
Val Loss: 1.3407 | Val Acc: 55.81%

Epoch [19/20]


Train Loss: 1.6398 | Train Acc: 45.56%
Val Loss: 1.3251 | Val Acc: 56.64%

Epoch [20/20]


Train Loss: 1.6226 | Train Acc: 45.94%
Val Loss: 1.2837 | Val Acc: 57.82%
✓ Best model saved: ..\models\efficientnet_frozen_best.pth

Best validation accuracy: 57.82% (Epoch 20)

Transfer learning training complete!
Best validation accuracy: 57.82%


In [4]:
import sys
from pathlib import Path
import torch

sys.path.insert(0, str(Path("../src").resolve()))

from brain_data import prepare_data_loaders
from models_tl import EfficientNetTransfer
from train import train_model

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load data
print("\n" + "="*60)
print("Loading data...")
print("="*60)
train_loader, val_loader, test_loader, class_weights, info = prepare_data_loaders(
    data_dir="../data/raw",
    batch_size=32,
    img_size=224,
)

# Create model with UNFROZEN backbone for fine-tuning
print("\nCreating EfficientNet-B0 model (fine-tuning mode)...")
model = EfficientNetTransfer(num_classes=info['num_classes'], freeze_backbone=False, dropout=0.5)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Train with lower learning rate for fine-tuning
print("\n" + "="*60)
print("Starting fine-tuning (unfrozen backbone)...")
print("="*60)

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=30,
    lr=5e-5,    # Very low learning rate for fine-tuning
    device=device,
    class_weights=class_weights,
    save_dir="../models",
    model_name="efficientnet_finetuned"
)

print("\nFine-tuning complete!")
print(f"Best validation accuracy: {max(history['val_acc']):.2f}%")

Using device: cuda

Loading data...
Found 30 classes
Astrocytoma T1                 |  792 images
Astrocytoma T1C+               |  882 images
Astrocytoma T2                 |  562 images
Ependymoma T1                  |  626 images
Ependymoma T1C+                |  748 images
Ependymoma T2                  |  700 images
Glioma T1                      | 1044 images
Glioma T1C+                    | 1098 images
Glioma T2                      |  810 images
Hemangiopericytoma T1          |  356 images
Hemangiopericytoma T1C+        |  614 images
Hemangiopericytoma T2          |  236 images
Meningioma T1                  | 1272 images
Meningioma T1C+                | 1954 images
Meningioma T2                  |  916 images
Neurocytoma T1                 |  380 images
Neurocytoma T1C+               |  508 images
Neurocytoma T2                 |  348 images
Normal T1                      |  830 images
Normal T1C+                    |  528 images
Normal T2                      |  758 images
Ol

c:\Users\anish\anaconda3\envs\gridlock\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(



Epoch [1/30]


Train Loss: 2.9986 | Train Acc: 15.16%
Val Loss: 2.1140 | Val Acc: 32.80%
✓ Best model saved: ..\models\efficientnet_finetuned_best.pth

Epoch [2/30]


Train Loss: 1.8622 | Train Acc: 38.12%
Val Loss: 1.2045 | Val Acc: 58.11%
✓ Best model saved: ..\models\efficientnet_finetuned_best.pth

Epoch [3/30]


...

Train Loss: 0.0615 | Train Acc: 98.01%
Val Loss: 0.0469 | Val Acc: 98.47%

Epoch [27/30]


Train Loss: 0.0612 | Train Acc: 98.02%
Val Loss: 0.0664 | Val Acc: 97.96%

Epoch [28/30]


Train Loss: 0.0558 | Train Acc: 97.99%
Val Loss: 0.0387 | Val Acc: 98.82%
✓ Best model saved: ..\models\efficientnet_finetuned_best.pth

Epoch [29/30]


Train Loss: 0.0545 | Train Acc: 98.12%
Val Loss: 0.0449 | Val Acc: 98.55%

Epoch [30/30]


Train Loss: 0.0537 | Train Acc: 98.19%
Val Loss: 0.0329 | Val Acc: 98.88%
✓ Best model saved: ..\models\efficientnet_finetuned_best.pth

Best validation accuracy: 98.88% (Epoch 30)

Fine-tuning complete!
Best validation accuracy: 98.88%


In [5]:
import subprocess
import sys

# Run the benchmark script
result = subprocess.run(
    [sys.executable, "../src/benchmark_gpu.py"],
    capture_output=True,
    text=True,
    cwd="../notebooks"
)

print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)


STDERR: c:\Users\anish\anaconda3\envs\gridlock\python.exe: can't open file 'e:\\programming\\Brain_MRI\\src\\benchmark_gpu.py': [Errno 2] No such file or directory



In [6]:
import sys
from pathlib import Path
import torch

sys.path.insert(0, str(Path("../src").resolve()))

from evaluate import evaluate_model, plot_confusion_matrix, plot_per_class_metrics, save_evaluation_report
from models_tl import EfficientNetTransfer
from brain_data import prepare_data_loaders

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Load test data
print("\nLoading test data...")
_, _, test_loader, _, info = prepare_data_loaders(
    data_dir="../data/raw",
    batch_size=32,
    img_size=224,
)

# Load best fine-tuned model
checkpoint_path = Path("../models/efficientnet_finetuned_best.pth")

if checkpoint_path.exists():
    print(f"Loading model from: {checkpoint_path}")
    model = EfficientNetTransfer(num_classes=info['num_classes'])
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    
    # Evaluate on test set
    print("\n" + "="*60)
    print("EVALUATING ON TEST SET")
    print("="*60)
    eval_results = evaluate_model(model, test_loader, device, info['class_names'])
    
    # Generate visualizations
    print("\nGenerating visualizations...")
    plot_confusion_matrix(eval_results)
    plot_per_class_metrics(eval_results)
    save_evaluation_report(eval_results)
    
    print(f"\n{'='*60}")
    print("✓ EVALUATION COMPLETE!")
    print("="*60)
    print(f"Test Accuracy: {eval_results['accuracy']*100:.2f}%")
    print(f"F1-Score (Macro): {eval_results['f1_macro']:.4f}")
    print(f"F1-Score (Weighted): {eval_results['f1_weighted']:.4f}")
    print("\nGenerated files:")
    print("  ✓ ../results/confusion_matrix.png")
    print("  ✓ ../results/per_class_metrics.png")
    print("  ✓ ../results/evaluation_report.txt")
    print(f"{'='*60}")
else:
    print(f"Model not found at {checkpoint_path}")

Device: cuda

Loading test data...
Found 30 classes
Astrocytoma T1                 |  792 images
Astrocytoma T1C+               |  882 images
Astrocytoma T2                 |  562 images
Ependymoma T1                  |  626 images
Ependymoma T1C+                |  748 images
Ependymoma T2                  |  700 images
Glioma T1                      | 1044 images
Glioma T1C+                    | 1098 images
Glioma T2                      |  810 images
Hemangiopericytoma T1          |  356 images
Hemangiopericytoma T1C+        |  614 images
Hemangiopericytoma T2          |  236 images
Meningioma T1                  | 1272 images
Meningioma T1C+                | 1954 images
Meningioma T2                  |  916 images
Neurocytoma T1                 |  380 images
Neurocytoma T1C+               |  508 images
Neurocytoma T2                 |  348 images
Normal T1                      |  830 images
Normal T1C+                    |  528 images
Normal T2                      |  758 images
Oli

C:\Users\anish\AppData\Local\Temp\ipykernel_8116\1371553402.py:28: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)



EVALUATING ON TEST SET


Evaluating: 100%|██████████| 106/106 [00:21<00:00,  4.93it/s]



TEST SET EVALUATION
Overall Accuracy: 98.97%
F1-Score (Macro): 0.9898
F1-Score (Weighted): 0.9897

PER-CLASS METRICS
                         precision    recall  f1-score   support

         Astrocytoma T1     0.9835    1.0000    0.9917       119
       Astrocytoma T1C+     0.9924    0.9924    0.9924       132
         Astrocytoma T2     0.9770    1.0000    0.9884        85
          Ependymoma T1     0.9570    0.9468    0.9519        94
        Ependymoma T1C+     0.9912    1.0000    0.9956       112
          Ependymoma T2     0.9510    0.9238    0.9372       105
              Glioma T1     1.0000    0.9745    0.9871       157
            Glioma T1C+     1.0000    1.0000    1.0000       165
              Glioma T2     0.9837    1.0000    0.9918       121
  Hemangiopericytoma T1     1.0000    1.0000    1.0000        54
Hemangiopericytoma T1C+     1.0000    1.0000    1.0000        92
  Hemangiopericytoma T2     1.0000    1.0000    1.0000        36
          Meningioma T1     0.9948  

In [7]:
import pandas as pd

# Model comparison
models_data = {
    'Model': ['Baseline CNN', 'EfficientNet (Frozen)', 'EfficientNet (Fine-tuned)'],
    'Training Time': ['68:38', '45:03', '~50 min'],
    'Epochs': [30, 20, 30],
    'Val Accuracy': ['34.34%', '57.82%', '98.88%'],
    'Test Accuracy': ['-', '-', '98.97%'],
    'Learning Rate': ['1e-3', '1e-4', '5e-5'],
}

df_models = pd.DataFrame(models_data)

print("="*80)
print("BRAIN TUMOR MRI CLASSIFICATION - PROJECT SUMMARY")
print("="*80)

print("\n1. MODEL PERFORMANCE COMPARISON")
print("-"*80)
print(df_models.to_string(index=False))

print("\n\n2. FINAL MODEL STATISTICS")
print("-"*80)
print(f"Architecture:        EfficientNet-B0 (Fine-tuned)")
print(f"Number of Classes:   30 brain tumor types")
print(f"Training Samples:    15,820")
print(f"Validation Samples:  3,390")
print(f"Test Samples:        3,390")
print(f"Image Resolution:    224 × 224 pixels")

print("\n\n3. TEST SET PERFORMANCE")
print("-"*80)
print(f"Overall Accuracy:    98.97%")
print(f"F1-Score (Macro):    0.9898")
print(f"F1-Score (Weighted): 0.9897")
print(f"Perfect Classes:     12 out of 30 (100% F1-score)")
print(f"Worst Class:         Ependymoma T2 (F1: 0.9372)")

print("\n\n4. HARDWARE USED")
print("-"*80)
print(f"GPU:                 NVIDIA RTX 4050 Laptop (6GB VRAM)")
print(f"CPU:                 Intel i5-13450HX")
print(f"PyTorch:             2.5.0 with CUDA 12.4")
print(f"Batch Size:          32 (optimal)")

print("\n\n5. KEY INSIGHTS")
print("-"*80)
print("""
✓ Transfer learning: 34% → 99% (+65% improvement)
✓ Fine-tuning > frozen backbone: 58% → 99% (+41% improvement)
✓ Class weighting handles imbalance well
✓ No overfitting: test ≈ validation accuracy
✓ All 30 classes perform excellently

Next steps:
  - Try ResNet-50 or MobileNet-V3 for comparison
  - Build ensemble of top models
  - Deploy to production (ONNX/TensorFlow Lite)
  - Validate on real hospital data
""")

print("="*80)

BRAIN TUMOR MRI CLASSIFICATION - PROJECT SUMMARY

1. MODEL PERFORMANCE COMPARISON
--------------------------------------------------------------------------------
                    Model Training Time  Epochs Val Accuracy Test Accuracy Learning Rate
             Baseline CNN         68:38      30       34.34%             -          1e-3
    EfficientNet (Frozen)         45:03      20       57.82%             -          1e-4
EfficientNet (Fine-tuned)       ~50 min      30       98.88%        98.97%          5e-5


2. FINAL MODEL STATISTICS
--------------------------------------------------------------------------------
Architecture:        EfficientNet-B0 (Fine-tuned)
Number of Classes:   30 brain tumor types
Training Samples:    15,820
Validation Samples:  3,390
Test Samples:        3,390
Image Resolution:    224 × 224 pixels


3. TEST SET PERFORMANCE
--------------------------------------------------------------------------------
Overall Accuracy:    98.97%
F1-Score (Macro):    0.98